In [2]:
using ForwardDiff
using Zygote
using NonlinearOptimizationTestFunctions

[ Info: Precompiling NonlinearOptimizationTestFunctions [3180b13f-e8d1-40aa-806b-3e6ce7522c6b] 
┌ Info: Skipping precompilation due to precompilable error. Importing NonlinearOptimizationTestFunctions [3180b13f-e8d1-40aa-806b-3e6ce7522c6b].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.


# Test Functions

In [130]:
using LinearAlgebra
# List all available functions (by their canonical lowercase name)
all_names = sort(collect(keys(TEST_FUNCTIONS))); println("Total functions: $(length(all_names))")
# Pick rosenbrock function n=10
rosen_tf = TEST_FUNCTIONS["rosenbrock"]  # Access via the dictionary key (the official :name)
tf10 = fixed(rosen_tf; n=10)
# start gives a challenging starting point
x = start(tf10); println("f(start): ", tf.f(x)) # Large positive value
# min_value and min_position give known global min
println("Global minimum: ", min_value(tf10), " at ", min_position(tf10))  # 0.0 at zeros
# Checking I know how to grab objective and gradient. Hessians are not provided. 
func = tf.f
grad = tf.grad
gradInPlace = tf.gradient!
y=similar(x)
gradInPlace(y,x)
norm(y-grad(x))
println("rosenbrock properties", properties(tf))

Total functions: 205
f(start): 5638.599999999999
Global minimum: 0.0 at [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
rosenbrock properties["bounded", "continuous", "differentiable", "ill-conditioned", "non-separable", "unimodal"]


I want to select smooth, non-separable, and scalable functions from the list.  This is easy enough and gets me down to 34 functions.

In [153]:
TestSuite = filter_testfunctions(tf -> differentiable(tf)&& !(separable(tf)) && scalable(tf))
tf.name
TestNames = getproperty.(TestSuite,:name);
join(TestNames,",")

"whitley,dixonprice,trid,xin_she_yang3,pathological,sphere_rotated,griewank,pinter,xin_she_yang4,schafferf6,sargan,powellsingular2,rosenbrock,shubert_shifted,trigonometric1,exponential,mishra11,ackley,mishra1,shubert_rotated,schwefel,zakharov,shubert_coupled,levyjamil,michalewicz,powellsingular,schwefel12,chungreynolds,mishra2,salomon,trigonometric2,rana,stretched_v_sine_wave,brown"

## Comparing LBFGS and BFGS and Newton in optimize using benchmark

In [57]:
using Optim, BenchmarkTools
# load rosenbrock n=10 run LBFGS with default settings but custom tolerances
n=20;
tf = fixed(TEST_FUNCTIONS["rosenbrock"]; n=n)
options = Optim.Options(g_tol = 1e-12, x_abstol = 1e-10, x_reltol=1e-10, iterations = 1000)
# pass options to optimize
result = optimize(tf.f, tf.gradient!, start(tf10), LBFGS(),options)
println("n=10: rosenbrock minimum = ", minimum(result))
println("n=10: rosenbrock minimizer = ", result.minimizer)
println("n=10: rosenbrock iterations = ", result.iterations)
@benchmark optimize(tf.f, tf.gradient!, x0, LBFGS(),options) setup=(x0=rand(n))

n=10: rosenbrock minimum = 2.968089155894057e-29
n=10: rosenbrock minimizer = [1.0, 1.0, 1.0000000000000002, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0000000000000002, 1.0000000000000004]
n=10: rosenbrock iterations = 64


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  125.400 μs …  15.153 ms  ┊ GC (min … max):  0.00% … 95.69%
 Time  (median):     287.500 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   316.062 μs ± 607.261 μs  ┊ GC (mean ± σ):  11.68% ±  5.96%

                        ▂▄▅▇██▇▆▂▁                               
  ▁▁▂▂▃▃▃▄▄▅▅▅▅▅▅▅▅▆▅▆▇███████████▇▅▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  125 μs           Histogram: frequency by time          518 μs <

 Memory estimate: 123.12 KiB, allocs estimate: 2961.

In [84]:
using Optim, BenchmarkTools
# load rosenbrock n=10 run LBFGS with default settings but custom tolerances
n=20;
tf = fixed(TEST_FUNCTIONS["rosenbrock"]; n=n)
options = Optim.Options(g_tol = 1e-12, x_abstol = 1e-10, x_reltol=1e-10, iterations = 1000)
# pass options to optimize
result = optimize(tf.f, tf.gradient!, start(tf10), LBFGS(),options)
println("n=10: rosenbrock minimum = ", minimum(result))
println("n=10: rosenbrock minimizer = ", result.minimizer)
println("n=10: rosenbrock iterations = ", result.iterations)
@benchmark optimize(tf.f, tf.gradient!, x0, BFGS(),options) setup=(x0=rand(n))

n=10: rosenbrock minimum = 2.968089155894057e-29
n=10: rosenbrock minimizer = [1.0, 1.0, 1.0000000000000002, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0000000000000002, 1.0000000000000004]
n=10: rosenbrock iterations = 64


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   94.400 μs …  12.833 ms  ┊ GC (min … max):  0.00% … 97.97%
 Time  (median):     188.000 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   217.608 μs ± 533.586 μs  ┊ GC (mean ± σ):  13.09% ±  5.26%

          ▁▁▂▃▄▆▅▇▆▅▆▅▅▄▆▆▆▆█▇█▇▇▆▆▆▃▂▁▁                         
  ▁▁▂▂▃▄▆▇██████████████████████████████▇▆▆▅▄▃▄▃▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁ ▅
  94.4 μs          Histogram: frequency by time          328 μs <

 Memory estimate: 110.10 KiB, allocs estimate: 2640.

In [109]:
x0=rand(n)
grad=tf.gradient!
grad(tf.f,x0)

LoadError: MethodError: no method matching copyto!(::typeof(rosenbrock), ::Vector{Float64})
The function `copyto!` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  copyto!([91m::Array{T}[39m, ::Array{T}) where T
[0m[90m   @[39m [90mBase[39m [90m[4marray.jl:322[24m[39m
[0m  copyto!([91m::Array[39m, ::Array)
[0m[90m   @[39m [90mBase[39m [90m[4marray.jl:319[24m[39m
[0m  copyto!([91m::SparseArrays.AbstractCompressedVector[39m, ::AbstractVector)
[0m[90m   @[39m [35mSparseArrays[39m [90mC:\Users\AllanStruthers\.julia\juliaup\julia-1.11.6+0.x64.w64.mingw32\share\julia\stdlib\v1.11\SparseArrays\src\[39m[90m[4msparsevector.jl:614[24m[39m
[0m  ...


In [105]:
n=100;
x0=Array(rand(n,1))
grad(x0)

LoadError: MethodError: no method matching (::OptimTestProblems.MultivariateProblems.var"#3#4"{…})(::Matrix{…})
The function `#3` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  (::OptimTestProblems.MultivariateProblems.var"#3#4")(::Any, [91m::Any[39m)
[0m[90m   @[39m [35mOptimTestProblems[39m [90mC:\Users\AllanStruthers\.julia\packages\OptimTestProblems\aDkMU\src\optim_tests\multivariate\[39m[90m[4mMultivariateProblems.jl:58[24m[39m


In [95]:
# The extended rosenborck function in OptimTestProblems has n=100.
# It does not have a Hessian!!! 
using Optim, OptimTestProblems
UP = UnconstrainedProblems
prob = UP.examples["Extended Rosenbrock"]
x0=prob.initial_x;
println("n=", length(x0))
func = UP.objective(prob)
grad = UP.gradient(prob)
hess = UP.hessian(prob)
res = optimize(func, grad, hess, prob.initial_x, Newton())

n=100


LoadError: Hessian not implemented for Extended Rosenbrock

In [70]:
using Optim, BenchmarkTools, OptimTestProblems
# load rosenbrock n=2 run Newton with default settings but custom tolerances
# the larger suite does not have hessians.
prob = UnconstrainedProblems.examples["Extended Rosenbrock"]

OptimTestProblems.MultivariateProblems.OptimizationProblem{OptimTestProblems.MultivariateProblems.UnconstrainedProblems.MatVecHolder{Vector{Float64}, Matrix{Float64}}, typeof(OptimTestProblems.MultivariateProblems.UnconstrainedProblems.extrosenbrock_fun_gradient!), Float64, String, Nothing}("Extended Rosenbrock (100)", OptimTestProblems.MultivariateProblems.UnconstrainedProblems.extrosenbrock, OptimTestProblems.MultivariateProblems.UnconstrainedProblems.extrosenbrock_gradient!, OptimTestProblems.MultivariateProblems.UnconstrainedProblems.extrosenbrock_fun_gradient!, OptimTestProblems.MultivariateProblems.UnconstrainedProblems.extrosenbrock_hessian!, nothing, [-1.2, -1.2, -1.2, -1.2, -1.2, -1.2, -1.2, -1.2, -1.2, -1.2  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 0.0, true, false, OptimTestProblems.MultivariateProblems.UnconstrainedProblems.MatVecHolder{Vector{Float64}, Matr

In [73]:
x0=rand(100);
func = prob.f
func(x0)

LoadError: MethodError: no method matching extrosenbrock(::Vector{Float64})
The function `extrosenbrock` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  extrosenbrock(::AbstractArray, [91m::OptimTestProblems.MultivariateProblems.UnconstrainedProblems.MatVecHolder[39m)
[0m[90m   @[39m [35mOptimTestProblems[39m [90mC:\Users\AllanStruthers\.julia\packages\OptimTestProblems\aDkMU\src\optim_tests\multivariate\[39m[90m[4mmore_testing.jl:12[24m[39m


In [69]:
options = Optim.Options(g_tol = 1e-12, x_abstol = 1e-10, x_reltol=1e-10, iterations = 1000)
result = optimize(prob.f, prob.g!, prob.h!, prob.initial_x, NewtonTrustRegion(),options)
result.minimizer

LoadError: MethodError: no method matching extrosenbrock_gradient!(::Vector{Float64}, ::Vector{Float64})
The function `extrosenbrock_gradient!` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  extrosenbrock_gradient!(::AbstractArray, ::AbstractArray, [91m::OptimTestProblems.MultivariateProblems.UnconstrainedProblems.MatVecHolder[39m)
[0m[90m   @[39m [35mOptimTestProblems[39m [90mC:\Users\AllanStruthers\.julia\packages\OptimTestProblems\aDkMU\src\optim_tests\multivariate\[39m[90m[4mmore_testing.jl:25[24m[39m


In [68]:
UnconstrainedProblems.examples

Dict{AbstractString, OptimTestProblems.MultivariateProblems.OptimizationProblem} with 17 entries:
  "Rosenbrock"               => OptimizationProblem{Nothing, Nothing, Float64, …
  "Quadratic Diagonal"       => OptimizationProblem{MatVecHolder{Vector{Float64…
  "Hosaki"                   => OptimizationProblem{Nothing, Nothing, Float64, …
  "Large Polynomial"         => OptimizationProblem{Nothing, Nothing, Float64, …
  "Penalty Function I"       => OptimizationProblem{ParaboloidStruct{Float64, M…
  "Beale"                    => OptimizationProblem{Nothing, Nothing, Float64, …
  "Extended Rosenbrock"      => OptimizationProblem{MatVecHolder{Vector{Float64…
  "Polynomial"               => OptimizationProblem{Nothing, Nothing, Float64, …
  "Powell"                   => OptimizationProblem{Nothing, Nothing, Float64, …
  "Exponential"              => OptimizationProblem{Nothing, Nothing, Float64, …
  "Paraboloid Diagonal"      => OptimizationProblem{ParaboloidStruct{Float64, S…
  "Parabolo

In [ ]:
# load rosenbrock n=10 run LBFGS with default settings but custom tolerances
options = Optim.Options(g_tol = 1e-12, x_abstol = 1e-10, x_reltol=1e-10, iterations = 1000)
# pass options to optimize
result = optimize(tf.f, tf.gradient!, start(tf10), LBFGS(),options)
println("n=10: rosenbrock minimum = ", minimum(result))
println("n=10: rosenbrock minimizer = ", result.minimizer)
println("n=10: rosenbrock iterations = ", result.iterations)
@benchmark optimize(tf.f, tf.gradient!, x0, BFGS(),options) setup=(x0=rand(n))

In [60]:
tf

TestFunction(NonlinearOptimizationTestFunctions.rosenbrock, NonlinearOptimizationTestFunctions.rosenbrock_gradient, NonlinearOptimizationTestFunctions.var"#4#8"{typeof(rosenbrock_gradient)}(NonlinearOptimizationTestFunctions.rosenbrock_gradient), NonlinearOptimizationTestFunctions.rosenbrock, NonlinearOptimizationTestFunctions.rosenbrock_gradient, Dict{Symbol, Any}(:description => "Rosenbrock function: sum_{i=1}^{n-1} [100 (x_{i+1} - x_i^2)^2 + (x_i - 1)^2]. Features a narrow parabolic valley, making it ill-conditioned and challenging for gradient-based methods. Properties based on Jamil & Yang (2013).", :start => NonlinearOptimizationTestFunctions.var"#12#13"{NonlinearOptimizationTestFunctions.var"#1258#1263", Int64}(NonlinearOptimizationTestFunctions.var"#1258#1263"(), 20), :lb => NonlinearOptimizationTestFunctions.var"#12#13"{NonlinearOptimizationTestFunctions.var"#1261#1266", Int64}(NonlinearOptimizationTestFunctions.var"#1261#1266"(), 20), :min_position => NonlinearOptimizationTes